In [ ]:
import sys
sys.path.append('/data_nfs/og86asub/netmap/netmap-evaluation/')

import scanpy as sc
import time 
import os.path as op
import os
import numpy as np
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.3f' % x)
import scipy.sparse as scs

import torch
import yaml


from netmap.utils.misc import write_config


from netmap.utils.data_utils import *
from netmap.utils.tf_utils import *
from netmap.utils.netmap_config import NetmapConfig

from netmap.model.train_model import create_model_zoo
#from netmap.grn.inferrence import inferrence
from src.data_simulation.data_simulation_config import DataSimulationConfig
from netmap.masking.internal import *
from netmap.masking.external import *

#import decoupler as dc
from netmap.masking.external import *
from netmap.downstream.edge_selection import *
from netmap.downstream.downstream import *
from compute_metrics import build_augmented_network, create_forward_reverse
import netmap.grn.inferrence as inferrence

import compute_metrics
import time

In [ ]:
net = 'net_84_10865_net_88_10937_net_90_11013'
dataset_config = f"/data_nfs/og86asub/netmap/netmap-evaluation/results/configurations/data_simulation/config_easy/{net}.config.yaml"
config = f'/data_nfs/og86asub/netmap/netmap-evaluation/results/netmap/config_9/config_easy/{net}/config.yaml'
outdir = f'/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_new/{net}'

os.makedirs(outdir, exist_ok=True)

dataset_config =DataSimulationConfig.read_yaml(dataset_config)
config = NetmapConfig.read_yaml(config)

# read network files
# Augmented net contains edges between genes which are controlled by the same transcription factor
nets = [(op.basename(op.dirname(filename)), pd.read_csv(op.join(filename), sep=dataset_config.separator)) for filename in dataset_config.edgelist]
off_net = [(op.basename(op.dirname(filename)), pd.read_csv(op.join(filename), sep=dataset_config.separator)) for filename in dataset_config.common_edges]
augmented_nets = [(net[0], build_augmented_network(net[1])) for net in nets]


#Create the networks for forward and reverse nets
forward_reverse_nets = [(net[0], create_forward_reverse(net[1])) for net in nets]
forward_reverse_nets_augmented = [(net[0], build_augmented_network(net[1])) for net in forward_reverse_nets]
forward_reverse_off = [(net[0], create_forward_reverse(net[1])) for net in off_net]

combined_net = [n[1] for n in nets] + [n[1] for n in off_net]
combined_net = np.concatenate(combined_net)
combined_net = pd.DataFrame(combined_net)
combined_net.columns = ['source', 'target']
combined_net['weight'] = 1

# load data
adata = sc.read_h5ad(config.input_data)

print(adata.shape)
## Get the data matrix from the CustumAnndata obeject

gene_names = np.array(adata.var.index)
model_start = time.monotonic()

if config.layer == 'counts':
    data_tensor = adata.layers['counts']
else:
    data_tensor = adata.X

if scs.issparse(data_tensor):
    data_tensor = torch.tensor(data_tensor.todense(), dtype=torch.float32)
else:
    data_tensor = torch.tensor(data_tensor, dtype=torch.float32)

start_time_all = time.monotonic()
hyper = {
    '64_1':[64], 
    '64_2':[64, 64],
    '64_3': [64, 64, 64],
    '32_1': [32], 
    '32_2': [32, 32], 
    '32_3': [32, 32, 32],
    '16_1' : [16],
    '16_2' : [16, 16],
    '16_3': [16, 16, 16]
}    

dropout_percentage = [0, 0.1]
model_type = ['NegativeBinomialAutoencoder', 'ZINBAutoencoder']

time_collector = {}

hd =  '64_1'
dp = 0
mt = 'NegativeBinomialAutoencoder'

start_training = time.monotonic() 
model_zoo = create_model_zoo(data_tensor,  n_models=1, n_epochs=config.epochs, model_type=mt, dropout_rate=dp, hidden_dim = hyper[hd] )
end_training = time.monotonic()

xai  = 'GuidedBackprop'
start_inference = time.monotonic()
grn_adata2 = inferrence.inferrence_model_wise(model_zoo, data_tensor.cuda(), gene_names, xai, n_models=[1])
end_inference = time.monotonic()
grn_adata2.obs['grn'] = pd.Categorical(adata.obs['grn'])
grn_ads = {'netmap_1':grn_adata2}
overlaps_ungrouped, collect_results = compute_metrics.compute_metrics(grn_ads=grn_ads, nets= nets, augmented_nets=augmented_nets, global_nets=off_net, group_key=dataset_config.group_key, group_by_target=False, aggregate=False)

collect_results['training_time'] = end_training - start_training
collect_results['inference_time'] = end_inference - start_inference


In [ ]:
def process_results(all_overlaps):
    all_overlaps['configuration'] = all_overlaps['method']+'_'+all_overlaps['layer']
    # this should be changed if we use different sized popultaitons
    all_overlaps['factor'] = all_overlaps['mean_cell_count']/500
    all_overlaps['factor'] = all_overlaps['factor'].apply(lambda x: min(1, x))
    all_overlaps['weighted_overlap'] = all_overlaps['avg_edges_recovered']*(all_overlaps['factor'])

    all_overlaps = all_overlaps[~all_overlaps.avg_edges_recovered.isna()]
    all_overlaps = all_overlaps[~all_overlaps.mean_cell_count.isna()]
    all_overlaps['percentage_overlap'] = all_overlaps['weighted_overlap']/all_overlaps['gold_standard_edges']

    all_overlaps['precision'] = all_overlaps['weighted_overlap']/(all_overlaps['avg_edges_recovered'] + (all_overlaps['max_possible_edges']*all_overlaps['n_top']))
    return all_overlaps

def compute_area_over_diagonal(group):
    # Ensure data is sorted by x-axis ('n_top') for correct trapezoidal integration
    group = group.sort_values(by='n_top')
    
    # Calculate the vertical distance from the diagonal: f(x) = y - x
    difference = group['percentage_overlap'] - group['n_top']
    
    # We only want the area where the line is above the diagonal (difference > 0).
    # Set negative differences to 0.
    positive_difference = np.maximum(difference, 0)
    
    # Use numpy's trapezoidal rule (numerical integration) to compute the area: 
    # Area = Integral of (y - x) dx over the positive region
    area = si.trapezoid(y=difference, x=group['n_top'])
    return area


In [ ]:
overlaps_ungrouped = process_results(overlaps_ungrouped)
area_results = overlaps_ungrouped.groupby(['net', 'target', 'direction', 'method', 'net_type', 'layer']).apply(compute_area_over_diagonal).rename('Area Over Diagonal')
area_results.reset_index()


,n_top,net,avg_edges_recovered,mean_cell_count,gold_standard_edges,max_possible_edges,target,direction,method,net_type,layer,configuration,factor,weighted_overlap,percentage_overlap,precision
2,0.050,net_84_10865,9,390.889,154,68644,on_target,forward,netmap_1,strict,X,netmap_1_X,0.782,7.036,0.046,0.002
3,0.100,net_84_10865,18,338.722,154,68644,on_target,forward,netmap_1,strict,X,netmap_1_X,0.677,12.194,0.079,0.002
4,0.200,net_84_10865,46,327.130,154,68644,on_target,forward,netmap_1,strict,X,netmap_1_X,0.654,30.096,0.195,0.002
5,0.250,net_84_10865,64,344.203,154,68644,on_target,forward,netmap_1,strict,X,netmap_1_X,0.688,44.058,0.286,0.003
6,0.500,net_84_10865,106,393.594,154,68644,on_target,forward,netmap_1,strict,X,netmap_1_X,0.787,83.442,0.542,0.002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,0.200,net_90_11013,85,318.494,216,68644,off_target,reverse,netmap_1,unspecific,X,netmap_1_X,0.637,54.144,0.251,0.004
32,0.250,net_90_11013,98,350.480,216,68644,off_target,reverse,netmap_1,unspecific,X,netmap_1_X,0.701,68.694,0.318,0.004
33,0.500,net_90_11013,157,367.580,216,68644,off_target,reverse,netmap_1,unspecific,X,netmap_1_X,0.735,115.420,0.534,0.003
34,0.750,net_90_11013,210,435.090,216,68644,off_target,reverse,netmap_1,unspecific,X,netmap_1_X,0.870,182.738,0.846,0.004


In [42]:
import scipy.integrate as si

In [43]:
2*3*3*3*2

108

In [48]:
op.join(outdir, f'{xai}_{hd}_{dp}_{mt}_clustering_score.json')


'/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_new/net_84_10865_net_88_10937_net_90_11013/GuidedBackprop_64_1_0_NegativeBinomialAutoencoder_clustering_score.json'